# nb02 v3: Augmentation vs. Hebbian Positive Pairs

Compares two ablations of CTLS training to determine which positive pair definition produces genuine multi-layer circuit organization.

- **Model A — Augmentation**: 100 epochs, InfoNCE on two augmented views of the same image. No class labels at any point.
- **Model B — Hebbian**: Phase 0 trains a standard ResNet18 with cross-entropy to convergence. Phase 1 mines trajectory-similar pairs from Phase 0 and continues with InfoNCE (Hebbian circuit reinforcement).

**Ground truth**: Phase 0 baseline trajectory similarities — independent of both models.

The nb01 class-label baseline is *not* trained here. Its numbers appear only in the final summary table as a reference.

In [ ]:
import os, sys, random, copy, math
from pathlib import Path
from collections import defaultdict

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import stats

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import CIFAR10
from tqdm.auto import tqdm

# Colab detection / Drive mount
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    if not os.path.exists('/content/trainable-circuits'):
        os.system('git clone <your-repo-url> /content/trainable-circuits')
    sys.path.insert(0, '/content/trainable-circuits')

try:
    from sklearn.cluster import KMeans
    from sklearn.metrics import silhouette_score, silhouette_samples
except ImportError:
    os.system('pip install scikit-learn -q')
    from sklearn.cluster import KMeans
    from sklearn.metrics import silhouette_score, silhouette_samples

try:
    import umap
    _UMAP_OK = True
except ImportError:
    os.system('pip install umap-learn -q')
    try:
        import umap
        _UMAP_OK = True
    except ImportError:
        _UMAP_OK = False
        print('WARNING: umap-learn not available; Section 9a will be skipped.')

from models.backbone import CTLSBackbone
from models.meta_encoder import MetaEncoder
from models.soft_mask import SoftMask
from losses.simclr import NTXentLoss
from data.ssl import MultiViewDataset, get_ssl_augmentation

# ── Paths ────────────────────────────────────────────────────────────────────
DRIVE_ROOT         = '/content/drive/MyDrive/ctls'
DATA_DIR           = '/content/data/cifar10'
PHASE0_CKPT        = f'{DRIVE_ROOT}/experiments/phase0_baseline/best.pt'
MODEL_A_CKPT       = f'{DRIVE_ROOT}/experiments/ablation_aug/best.pt'
MODEL_B_PHASE1     = f'{DRIVE_ROOT}/experiments/ablation_hebbian/phase1.pt'
MODEL_B_CKPT       = f'{DRIVE_ROOT}/experiments/ablation_hebbian/best.pt'
TRAJ_CLUSTER_INDEX = f'{DRIVE_ROOT}/experiments/ablation_hebbian/cluster_index.pt'
NB01_CKPT          = f'{DRIVE_ROOT}/experiments/unified_a/best.pt'

for _d in [
    f'{DRIVE_ROOT}/experiments/phase0_baseline',
    f'{DRIVE_ROOT}/experiments/ablation_aug',
    f'{DRIVE_ROOT}/experiments/ablation_hebbian',
    DATA_DIR,
]:
    Path(_d).mkdir(parents=True, exist_ok=True)

# ── Constants ────────────────────────────────────────────────────────────────
CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD  = (0.2470, 0.2435, 0.2616)
DEVICE     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEED       = 42
L          = 8   # ResNet18 trajectory layers
N_CLASSES  = 10

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f'Device: {DEVICE}  |  PyTorch: {torch.__version__}')

In [ ]:
# 1a. Transforms
val_tf = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
])

ssl_aug = get_ssl_augmentation()  # SimCLR-style: RandomResizedCrop + flip + ColorJitter + GaussianBlur

train_tf = transforms.Compose([   # mild augmentation for Hebbian cluster pairs
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
])

print('Transforms ready: val_tf | ssl_aug | train_tf')

## Section 1b — Model Factories

- `build_baseline(device)` — `CTLSBackbone` only; used for Phase 0 cross-entropy training and trajectory extraction.
- `build_model_a(device)` — `CTLSBackbone` + `MetaEncoder(weighted_sum)`; Model A (Augmentation).
- `build_model_b(device)` — `CTLSBackbone` + `RoPEMetaEncoder`; Model B (Hebbian). Identical to `transformer_cls` except RoPE replaces sinusoidal depth encoding.

In [ ]:
class RoPEMetaEncoder(nn.Module):
    # Transformer-CLS meta-encoder with Rotary Position Embedding (RoPE).
    # Identical to MetaEncoder(encoder_type='transformer_cls') except
    # sinusoidal pos_enc is replaced with per-token rotation applied after
    # projection, before the transformer.  CLS token receives no rotation.
    def __init__(self, layer_dims, embedding_dim=64, projection_dim=128):
        super().__init__()
        self.L = len(layer_dims)
        D = projection_dim
        self.projectors = nn.ModuleList([
            nn.Sequential(
                nn.Linear(d, D),
                nn.LayerNorm(D),
                nn.GELU(),
            ) for d in layer_dims
        ])
        self.cls_token = nn.Parameter(torch.zeros(1, 1, D))
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        # Precompute RoPE cos/sin tables: shape [L, D//2]
        half = D // 2
        rope_cos = torch.zeros(self.L, half)
        rope_sin = torch.zeros(self.L, half)
        for l in range(self.L):
            for i in range(half):
                theta = l * (10000 ** (-2 * i / D))
                rope_cos[l, i] = math.cos(theta)
                rope_sin[l, i] = math.sin(theta)
        self.register_buffer('rope_cos', rope_cos)
        self.register_buffer('rope_sin', rope_sin)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=D, nhead=4, dim_feedforward=D*2,
            dropout=0.0, batch_first=True, norm_first=True,
        )
        self.transformer = nn.TransformerEncoder(enc_layer, num_layers=2)
        self.readout = nn.Linear(D, embedding_dim)

    def _apply_rope(self, x):
        B, Lx, D = x.shape
        xp = x.view(B, Lx, D // 2, 2)        # [B, L, half, 2]
        x0, x1 = xp[..., 0], xp[..., 1]      # [B, L, half] each
        cos = self.rope_cos.unsqueeze(0)       # [1, L, half]
        sin = self.rope_sin.unsqueeze(0)
        rot = torch.stack([x0*cos - x1*sin, x0*sin + x1*cos], dim=-1)
        return rot.view(B, Lx, D)

    def forward(self, trajectory):
        B = trajectory[0].shape[0]
        proj = [p(h) for p, h in zip(self.projectors, trajectory)]
        x = torch.stack(proj, dim=1)            # [B, L, D]
        x = self._apply_rope(x)                 # apply RoPE to layer tokens
        cls = self.cls_token.expand(B, -1, -1)  # [B, 1, D]
        tokens = torch.cat([cls, x], dim=1)     # [B, L+1, D]
        enc = self.transformer(tokens)
        z = self.readout(enc[:, 0])             # CLS token
        return F.normalize(z, dim=-1)


def build_baseline(device):
    sm = SoftMask(init_temperature=1.0)
    bb = CTLSBackbone(arch='resnet18', num_classes=10, pretrained=False, soft_mask=sm).to(device)
    return bb


def build_model_a(device):
    sm = SoftMask(init_temperature=1.0)
    bb = CTLSBackbone(arch='resnet18', num_classes=10, pretrained=False, soft_mask=sm).to(device)
    me = MetaEncoder(
        layer_dims=bb.layer_dims, embedding_dim=64,
        encoder_type='weighted_sum', projection_dim=128,
    ).to(device)
    return bb, me, sm


def build_model_b(device):
    sm = SoftMask(init_temperature=1.0)
    bb = CTLSBackbone(arch='resnet18', num_classes=10, pretrained=False, soft_mask=sm).to(device)
    me = RoPEMetaEncoder(
        layer_dims=bb.layer_dims, embedding_dim=64, projection_dim=128,
    ).to(device)
    return bb, me, sm


print('Model factories ready. layer_dims:', build_baseline(DEVICE).layer_dims)

## Section 1c — Training Utilities

In [ ]:
def get_lambda(epoch, warmup=10):
    return min(1.0, epoch / warmup)


def get_tau(epoch, init=1.0, final=0.1, anneal=80):
    if epoch >= anneal:
        return final
    t = epoch / anneal
    return final + 0.5 * (init - final) * (1 + math.cos(math.pi * t))


def train_epoch_aug(backbone, meta_enc, loader, optimizer, infonce, lam, device):
    backbone.train(); meta_enc.train()
    tot_loss = tot_info = n = 0
    for v1, v2, _ in loader:
        v1, v2 = v1.to(device), v2.to(device)
        _, traj1 = backbone(v1)
        _, traj2 = backbone(v2)
        z1 = meta_enc(traj1)
        z2 = meta_enc(traj2)
        info = infonce(z1, z2)
        loss = lam * info
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        tot_loss += loss.item(); tot_info += info.item(); n += 1
    return tot_loss / n, tot_info / n


def train_epoch_baseline(backbone, loader, optimizer, device):
    backbone.train()
    tot_loss = tot_correct = total = n = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        logits, _ = backbone(x)
        loss = F.cross_entropy(logits, y)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        tot_loss += loss.item()
        tot_correct += (logits.argmax(-1) == y).sum().item()
        total += y.size(0); n += 1
    return tot_loss / n, tot_correct / total


def train_epoch_hebbian(backbone, meta_enc, loader, optimizer, infonce, lam, device):
    backbone.train(); meta_enc.train()
    tot_loss = tot_info = n = 0
    for img1, img2, _ in loader:
        img1, img2 = img1.to(device), img2.to(device)
        _, traj1 = backbone(img1)
        _, traj2 = backbone(img2)
        z1 = meta_enc(traj1)
        z2 = meta_enc(traj2)
        info = infonce(z1, z2)
        loss = lam * info
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        tot_loss += loss.item(); tot_info += info.item(); n += 1
    return tot_loss / n, tot_info / n


def eval_acc(backbone, root, device, batch_size=256):
    backbone.eval()
    ds = CIFAR10(root=root, train=False, transform=val_tf, download=False)
    dl = DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=0)
    correct = total = 0
    with torch.no_grad():
        for x, y in dl:
            logits, _ = backbone(x.to(device))
            correct += (logits.argmax(-1) == y.to(device)).sum().item()
            total += y.size(0)
    return correct / total


def eval_knn_acc(backbone, root, device, k=5, ref_size=2048, batch_size=256):
    # k-NN accuracy on last trajectory point (avgpool features).
    # Use for SSL-trained models where the fc head has no class gradient.
    backbone.eval()
    def _feats(ds, max_n=None):
        dl = DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=0)
        fs, ls = [], []
        n = 0
        with torch.no_grad():
            for x, y in dl:
                _, traj = backbone(x.to(device))
                fs.append(F.normalize(traj[-1], dim=-1).cpu())
                ls.append(y)
                n += y.size(0)
                if max_n and n >= max_n:
                    break
        f = torch.cat(fs); l = torch.cat(ls)
        return (f[:max_n], l[:max_n]) if max_n else (f, l)
    tr = CIFAR10(root=root, train=True,  transform=val_tf, download=False)
    vl = CIFAR10(root=root, train=False, transform=val_tf, download=False)
    rf, rl = _feats(tr, max_n=ref_size)
    vf, vl2 = _feats(vl)
    sims = vf @ rf.T
    topk = sims.topk(k, dim=1).indices
    pred = rl[topk].mode(dim=1).values
    return (pred == vl2).float().mean().item()


def save_ckpt_eval(path, backbone, meta_enc, val_acc):
    d = {'val_acc': val_acc, 'backbone': backbone.state_dict()}
    if meta_enc is not None:
        d['meta_enc'] = meta_enc.state_dict()
    torch.save(d, path)


def save_ckpt(path, backbone, meta_enc, optimizer, epoch, val_acc):
    d = {'epoch': epoch, 'val_acc': val_acc,
         'backbone': backbone.state_dict(),
         'optimizer': optimizer.state_dict()}
    if meta_enc is not None:
        d['meta_enc'] = meta_enc.state_dict()
    torch.save(d, path)


def load_ckpt(path, backbone, meta_enc=None, optimizer=None, device=None):
    ckpt = torch.load(path, map_location=device or 'cpu')
    backbone.load_state_dict(ckpt['backbone'])
    if meta_enc is not None and 'meta_enc' in ckpt:
        meta_enc.load_state_dict(ckpt['meta_enc'])
    if optimizer is not None and 'optimizer' in ckpt:
        optimizer.load_state_dict(ckpt['optimizer'])
    va = ckpt.get('val_acc', float('nan'))
    ep = ckpt.get('epoch', '?')
    print(f'Loaded {path}  (epoch={ep}, val_acc={va:.4f})')
    return ckpt.get('epoch', 0)


def bootstrap_spearmanr(x, y, n_boot=2000, ci=0.95, seed=0):
    rng = np.random.default_rng(seed)
    x, y = np.asarray(x, dtype=float), np.asarray(y, dtype=float)
    n = len(x)
    rhos = [stats.spearmanr(x[idx := rng.integers(0, n, n)], y[idx])[0]
            for _ in range(n_boot)]
    alpha = 1 - ci
    lo, hi = np.percentile(rhos, [100*alpha/2, 100*(1-alpha/2)])
    return float(np.mean(rhos)), lo, hi


def permutation_test_delta_rho(x1, y, x2, n_perm=2000, seed=1):
    x1 = np.asarray(x1, dtype=float)
    x2 = np.asarray(x2, dtype=float)
    y  = np.asarray(y,  dtype=float)
    rng = np.random.default_rng(seed)
    obs = stats.spearmanr(x2, y)[0] - stats.spearmanr(x1, y)[0]
    n = len(y)
    null = []
    for _ in range(n_perm):
        p = rng.permutation(n)
        null.append(stats.spearmanr(x2[p], y)[0] - stats.spearmanr(x1[p], y)[0])
    null = np.array(null)
    return obs, float((null >= obs).mean()), null


def compute_pair_sims(backbone, meta_enc, val_ds, pairs, device, batch_size=256):
    backbone.eval()
    if meta_enc is not None:
        meta_enc.eval()
    all_idx = sorted(set(i for p in pairs for i in p))
    pos_map = {idx: pos for pos, idx in enumerate(all_idx)}
    sub = torch.utils.data.Subset(val_ds, all_idx)
    dl  = DataLoader(sub, batch_size=batch_size, shuffle=False, num_workers=0)
    layer_acts = [[] for _ in range(L)]
    z_list = []
    with torch.no_grad():
        for x, _ in dl:
            _, traj = backbone(x.to(device))
            for l in range(L):
                layer_acts[l].append(traj[l].cpu())
            if meta_enc is not None:
                z_list.append(meta_enc(traj).cpu())
    layer_acts = [torch.cat(a) for a in layer_acts]
    z_all = torch.cat(z_list) if z_list else None
    n_pairs = len(pairs)
    lsims  = np.zeros((n_pairs, L))
    zsims  = np.zeros(n_pairs) if z_all is not None else None
    h8sims = np.zeros(n_pairs)
    for k, (i, j) in enumerate(pairs):
        pi, pj = pos_map[i], pos_map[j]
        for l in range(L):
            hi = F.normalize(layer_acts[l][pi].unsqueeze(0), dim=-1)
            hj = F.normalize(layer_acts[l][pj].unsqueeze(0), dim=-1)
            lsims[k, l] = (hi @ hj.T).item()
        if z_all is not None:
            zsims[k] = (z_all[pi] @ z_all[pj]).item()
        hi8 = F.normalize(layer_acts[-1][pi].unsqueeze(0), dim=-1)
        hj8 = F.normalize(layer_acts[-1][pj].unsqueeze(0), dim=-1)
        h8sims[k] = (hi8 @ hj8.T).item()
    return lsims, zsims, h8sims


print('Utilities ready.')

## Section 2 — Model A: Augmentation Training (100 epochs)

Model A tests whether removing class labels from the positive pair definition breaks the z ≈ h₈ collapse seen in nb01. Two augmented views of the same image are a valid positive pair by construction — no class signal at any point.

**Checkpoint criterion**: k-NN accuracy on last trajectory features (the ResNet18 fc head receives no gradient from InfoNCE, so logit accuracy is ~10% regardless of representation quality).

In [ ]:
SKIP_A = os.path.exists(MODEL_A_CKPT)
AUG_EPOCHS = 100

backbone_a, meta_enc_a, soft_mask_a = build_model_a(DEVICE)

if SKIP_A:
    load_ckpt(MODEL_A_CKPT, backbone_a, meta_enc_a, device=DEVICE)
    print('Loaded Model A checkpoint.')
else:
    params_a    = list(backbone_a.parameters()) + list(meta_enc_a.parameters())
    opt_a       = AdamW(params_a, lr=1e-3, weight_decay=1e-4)
    sched_a     = CosineAnnealingLR(opt_a, T_max=AUG_EPOCHS, eta_min=1e-5)
    infonce_a   = NTXentLoss(temperature=0.07)
    train_ds_a  = MultiViewDataset(
        root=DATA_DIR, train=True, augmentation=get_ssl_augmentation(),
        dataset_name='cifar10', download=True,
    )
    train_dl_a  = DataLoader(train_ds_a, batch_size=256, shuffle=True,
                             drop_last=True, num_workers=0)
    best_acc_a  = 0.0
    print(f'Training Model A: {AUG_EPOCHS} epochs, augmentation-based positive pairs')
    for ep in range(1, AUG_EPOCHS + 1):
        lam = get_lambda(ep - 1)
        tau = get_tau(ep - 1)
        soft_mask_a.set_temperature(tau)
        loss, info = train_epoch_aug(
            backbone_a, meta_enc_a, train_dl_a, opt_a, infonce_a, lam, DEVICE)
        sched_a.step()
        acc = eval_knn_acc(backbone_a, DATA_DIR, DEVICE)
        print(f'  Ep {ep:3d} | loss={loss:.4f} info={info:.4f} | '
              f'knn_acc={acc:.4f} | lam={lam:.3f} tau={tau:.3f}')
        if acc > best_acc_a:
            best_acc_a = acc
            save_ckpt_eval(MODEL_A_CKPT, backbone_a, meta_enc_a, acc)
    print(f'Training complete. Best k-NN acc = {best_acc_a:.4f}')
    load_ckpt(MODEL_A_CKPT, backbone_a, meta_enc_a, device=DEVICE)

backbone_a.eval(); meta_enc_a.eval()
print('Model A ready.')

## Section 3 — Model B: Hebbian Clustering

**Motivation**: Hebbian learning — *neurons that fire together wire together*. Applied at the circuit level: if two images already co-activate similar internal circuits (Phase 0 trajectory similarity), Phase 1 explicitly reinforces that grouping through InfoNCE.

**Key difference from nb02 v1**: The trajectory clusters are derived from a model trained with *zero* CTLS pressure (Phase 0 cross-entropy only). This means the trajectory space reflects natural feature organization, not class-label contrastive collapse. Neither Model A nor Model B sees Phase 0 trajectory space during their respective CTLS phases — so Phase 0 similarities are a clean independent ground truth.

**Three sub-steps**:
- 3a. Train Phase 0 baseline (ResNet18, cross-entropy, 100 epochs)
- 3b. Extract trajectories from 10k training images; k-means cluster them (k=100)
- 3c. Phase 1 CTLS training using cluster membership as positive pairs (70 epochs)

### 3a — Phase 0: Baseline ResNet18

Train a standard ResNet18 with cross-entropy only. This is the **discovery phase** — the model finds what internal representations are useful without any circuit consistency pressure. Phase 0 weights initialize Model B's backbone in Phase 1.

In [ ]:
SKIP_PHASE0   = os.path.exists(PHASE0_CKPT)
PHASE0_EPOCHS = 100

backbone_p0 = build_baseline(DEVICE)

if SKIP_PHASE0:
    load_ckpt(PHASE0_CKPT, backbone_p0, device=DEVICE)
    print('Loaded Phase 0 checkpoint.')
else:
    opt_p0   = AdamW(backbone_p0.parameters(), lr=1e-3, weight_decay=1e-4)
    sched_p0 = CosineAnnealingLR(opt_p0, T_max=PHASE0_EPOCHS, eta_min=1e-5)
    train_ds_p0 = CIFAR10(root=DATA_DIR, train=True, transform=train_tf, download=True)
    train_dl_p0 = DataLoader(train_ds_p0, batch_size=256, shuffle=True,
                             drop_last=True, num_workers=0)
    best_acc_p0 = 0.0
    print(f'Training Phase 0 baseline: {PHASE0_EPOCHS} epochs, cross-entropy only')
    for ep in range(1, PHASE0_EPOCHS + 1):
        loss, acc = train_epoch_baseline(backbone_p0, train_dl_p0, opt_p0, DEVICE)
        sched_p0.step()
        val_acc_p0 = eval_acc(backbone_p0, DATA_DIR, DEVICE)
        print(f'  Ep {ep:3d} | train_loss={loss:.4f} train_acc={acc:.4f} | val_acc={val_acc_p0:.4f}')
        if val_acc_p0 > best_acc_p0:
            best_acc_p0 = val_acc_p0
            save_ckpt_eval(PHASE0_CKPT, backbone_p0, None, val_acc_p0)
    print(f'Phase 0 complete. Best val acc = {best_acc_p0:.4f}')
    load_ckpt(PHASE0_CKPT, backbone_p0, device=DEVICE)

backbone_p0.eval()
p0_acc = eval_acc(backbone_p0, DATA_DIR, DEVICE)
print(f'Phase 0 final val_acc = {p0_acc:.4f}')

### 3b — Trajectory Extraction and Clustering

Extract activation trajectories from the Phase 0 model for 10,000 training images. Concatenate all 8 layer activations (L2-normalized per layer) into a 1920-dim vector. Run k-means (k=100) to find natural circuit groups. These clusters define the Hebbian positive pairs — images whose circuits already fire together.

**Cluster diagnostics** reveal whether clusters capture within-class sub-structure (desired) or simply rediscover class identity (which would degrade Model B to nb01).

In [ ]:
SKIP_CLUSTERING = os.path.exists(TRAJ_CLUSTER_INDEX)
N_CLUSTER_IMGS  = 10_000
N_CLUSTERS      = 100

if SKIP_CLUSTERING:
    clust_data = torch.load(TRAJ_CLUSTER_INDEX, map_location='cpu')
    cluster_labels_arr   = clust_data['cluster_labels']
    image_indices_arr    = clust_data['image_indices']
    cluster_id_per_image = clust_data['cluster_id_per_image']
    print(f'Loaded cluster index: {N_CLUSTER_IMGS} images, {N_CLUSTERS} clusters.')
else:
    rng_clust = np.random.default_rng(SEED + 10)
    full_train = CIFAR10(root=DATA_DIR, train=True, transform=None, download=False)
    image_indices_arr = rng_clust.choice(len(full_train), N_CLUSTER_IMGS, replace=False)
    image_indices_arr = np.sort(image_indices_arr)

    sub_ds = torch.utils.data.Subset(
        CIFAR10(root=DATA_DIR, train=True, transform=val_tf, download=False),
        image_indices_arr.tolist(),
    )
    sub_dl = DataLoader(sub_ds, batch_size=256, shuffle=False, num_workers=0)

    traj_vecs = []
    class_lbls_sub = []
    raw_norms = [[] for _ in range(L)]
    print('Extracting trajectories from Phase 0 model...')
    backbone_p0.eval()
    with torch.no_grad():
        for x, y in tqdm(sub_dl):
            _, traj = backbone_p0(x.to(DEVICE))
            for l in range(L):
                raw_norms[l].append(traj[l].norm(dim=-1).mean().item())
            vec = torch.cat([F.normalize(traj[l], dim=-1) for l in range(L)], dim=-1)
            traj_vecs.append(vec.cpu())
            class_lbls_sub.extend(y.tolist())
    traj_vecs = torch.cat(traj_vecs).numpy()  # [10000, 1920]
    class_lbls_sub = np.array(class_lbls_sub)

    print(f'Trajectory matrix shape: {traj_vecs.shape}')
    print('Per-layer mean activation norm (pre-normalization sanity check):')
    for l in range(L):
        print(f'  Layer {l+1}: {np.mean(raw_norms[l]):.3f}')

    print(f'Running k-means (k={N_CLUSTERS})...')
    km = KMeans(n_clusters=N_CLUSTERS, n_init=10, max_iter=300, random_state=SEED)
    cluster_id_per_image = km.fit_predict(traj_vecs)

    # ── Cluster diagnostics ───────────────────────────────────────────────────
    sizes = np.bincount(cluster_id_per_image)
    print(f'Cluster sizes: min={sizes.min()} max={sizes.max()} mean={sizes.mean():.1f} std={sizes.std():.1f}')

    purities = []
    for cid in range(N_CLUSTERS):
        mask = cluster_id_per_image == cid
        if mask.sum() == 0:
            continue
        lbls_in = class_lbls_sub[mask]
        modal_frac = np.bincount(lbls_in, minlength=N_CLASSES).max() / mask.sum()
        purities.append(modal_frac)
    mean_purity = np.mean(purities)
    print(f'Cluster mean class purity: {mean_purity:.3f}')
    if mean_purity > 0.85:
        print('WARNING: Clusters are class-dominated. '
              'Hebbian pairs may degrade to class-label pairs.')
    else:
        print('Clusters show within-class diversity (purity <= 0.85) -- good.')

    fig, ax = plt.subplots(figsize=(7, 3))
    ax.hist(purities, bins=20, color='steelblue', edgecolor='white')
    ax.axvline(0.85, color='red', linestyle='--', label='0.85 warning threshold')
    ax.set_xlabel('Cluster class purity'); ax.set_ylabel('Count')
    ax.set_title(f'Cluster purity distribution (mean={mean_purity:.2f})')
    ax.legend()
    plt.tight_layout(); plt.show()

    # Per-layer contribution: variance of centroids along each layer's dims
    centroids = km.cluster_centers_  # [100, 1920]
    layer_starts = np.cumsum([0] + [64, 64, 128, 128, 256, 256, 512, 512])
    print('Per-layer centroid variance (higher = layer drives clustering more):')
    for l in range(L):
        s, e = layer_starts[l], layer_starts[l+1]
        var = centroids[:, s:e].var(axis=0).mean()
        print(f'  Layer {l+1}: {var:.5f}')

    cluster_labels_arr = km.labels_
    torch.save({
        'cluster_labels':   cluster_labels_arr,
        'image_indices':    image_indices_arr,
        'cluster_id_per_image': cluster_id_per_image,
    }, TRAJ_CLUSTER_INDEX)
    print(f'Cluster index saved to {TRAJ_CLUSTER_INDEX}')

print(f'Clustering done: {N_CLUSTER_IMGS} images, {N_CLUSTERS} clusters.')

In [ ]:
class ClusterPairedDataset(torch.utils.data.Dataset):
    # Returns (img1, img2, cluster_id) where img1/img2 are from the same
    # trajectory cluster.  Handles single-member clusters by falling back
    # to a random same-class pair.
    def __init__(self, root, image_indices, cluster_id_per_image, transform):
        self.dataset = CIFAR10(root=root, train=True, transform=None, download=False)
        self.img_idx = list(image_indices)  # positions into full CIFAR-10 train set
        self.cluster_ids = list(cluster_id_per_image)
        self.transform = transform
        ctp = defaultdict(list)
        for pos, cid in enumerate(self.cluster_ids):
            ctp[cid].append(pos)
        self.cluster_to_pos = dict(ctp)
        self.labels = [self.dataset.targets[idx] for idx in self.img_idx]
        ctp2 = defaultdict(list)
        for pos, lbl in enumerate(self.labels):
            ctp2[lbl].append(pos)
        self.class_to_pos = dict(ctp2)

    def __len__(self):
        return len(self.img_idx)

    def __getitem__(self, pos):
        cid = self.cluster_ids[pos]
        members = self.cluster_to_pos[cid]
        if len(members) > 1:
            candidates = [p for p in members if p != pos]
            other = random.choice(candidates)
        else:
            lbl = self.labels[pos]
            same_cls = [p for p in self.class_to_pos[lbl] if p != pos]
            other = random.choice(same_cls) if same_cls else pos
        img1_pil, _ = self.dataset[self.img_idx[pos]]
        img2_pil, _ = self.dataset[self.img_idx[other]]
        return self.transform(img1_pil), self.transform(img2_pil), cid


print('ClusterPairedDataset defined.')

### 3d — Phase 1: Hebbian CTLS Training (70 epochs)

Continue from Phase 0 weights. InfoNCE uses cluster-derived positive pairs. Lower learning rate (5e-4 vs 1e-3) because Phase 0 weights are already converged — high lr would destroy learned representations. λ warmup over 10 epochs introduces circuit consistency pressure gradually.

In [ ]:
SKIP_B       = os.path.exists(MODEL_B_CKPT)
PHASE1_EPOCHS = 70

backbone_b, meta_enc_b, soft_mask_b = build_model_b(DEVICE)

if SKIP_B:
    load_ckpt(MODEL_B_CKPT, backbone_b, meta_enc_b, device=DEVICE)
    print('Loaded Model B checkpoint.')
else:
    # Initialize backbone from Phase 0
    p0_state = torch.load(PHASE0_CKPT, map_location=DEVICE)
    backbone_b.load_state_dict(p0_state['backbone'])
    print('Initialized backbone_b from Phase 0 weights.')

    params_b   = list(backbone_b.parameters()) + list(meta_enc_b.parameters())
    opt_b      = AdamW(params_b, lr=5e-4, weight_decay=1e-4)
    sched_b    = CosineAnnealingLR(opt_b, T_max=PHASE1_EPOCHS, eta_min=1e-5)
    infonce_b  = NTXentLoss(temperature=0.07)
    train_ds_b = ClusterPairedDataset(
        root=DATA_DIR,
        image_indices=image_indices_arr,
        cluster_id_per_image=cluster_id_per_image,
        transform=train_tf,
    )
    train_dl_b = DataLoader(train_ds_b, batch_size=256, shuffle=True,
                            drop_last=True, num_workers=0)
    best_acc_b = 0.0
    print(f'Training Model B Phase 1: {PHASE1_EPOCHS} epochs, Hebbian cluster pairs')
    for ep in range(1, PHASE1_EPOCHS + 1):
        lam = get_lambda(ep - 1)
        tau = get_tau(ep - 1)
        soft_mask_b.set_temperature(tau)
        loss, info = train_epoch_hebbian(
            backbone_b, meta_enc_b, train_dl_b, opt_b, infonce_b, lam, DEVICE)
        sched_b.step()
        acc = eval_acc(backbone_b, DATA_DIR, DEVICE)
        print(f'  Ep {ep:3d} | loss={loss:.4f} info={info:.4f} | '
              f'val_acc={acc:.4f} | lam={lam:.3f} tau={tau:.3f}')
        if acc > best_acc_b:
            best_acc_b = acc
            save_ckpt_eval(MODEL_B_CKPT, backbone_b, meta_enc_b, acc)
    save_ckpt(MODEL_B_PHASE1, backbone_b, meta_enc_b, opt_b, PHASE1_EPOCHS, best_acc_b)
    print(f'Phase 1 complete. Best val acc = {best_acc_b:.4f}')
    load_ckpt(MODEL_B_CKPT, backbone_b, meta_enc_b, device=DEVICE)

backbone_b.eval(); meta_enc_b.eval()
print('Model B ready.')

In [ ]:
# Section 4 — Load reference models
# Phase 0 backbone is already loaded (backbone_p0).  Attempt to load nb01.

backbone_p0.eval()

nb01_acc = None
backbone_nb01 = None
meta_enc_nb01 = None
if os.path.exists(NB01_CKPT):
    backbone_nb01, meta_enc_nb01, _ = build_model_a(DEVICE)
    load_ckpt(NB01_CKPT, backbone_nb01, meta_enc_nb01, device=DEVICE)
    backbone_nb01.eval(); meta_enc_nb01.eval()
    nb01_acc = eval_acc(backbone_nb01, DATA_DIR, DEVICE)
    print(f'nb01 Option A loaded. val_acc = {nb01_acc:.4f}')
else:
    print(f'nb01 checkpoint not found at {NB01_CKPT}. Reference column will use hardcoded values.')

p0_acc_ref   = eval_acc(backbone_p0,   DATA_DIR, DEVICE)
acc_a        = eval_knn_acc(backbone_a, DATA_DIR, DEVICE)
acc_b        = eval_acc(backbone_b,    DATA_DIR, DEVICE)

print()
print('Val accuracy summary:')
print(f'  Phase 0 baseline : {p0_acc_ref:.4f}')
print(f'  Model A (Aug)    : {acc_a:.4f}  [k-NN; no class signal]')
print(f'  Model B (Hebbian): {acc_b:.4f}')
if nb01_acc is not None:
    print(f'  nb01 reference   : {nb01_acc:.4f}')

In [ ]:
# Section 5 — Validation Dataset and Pair Construction
N_PAIRS_EACH = 500   # same-class + diff-class

val_ds_raw  = CIFAR10(root=DATA_DIR, train=False, transform=val_tf, download=False)
val_labels  = np.array(val_ds_raw.targets)

rng_pairs   = np.random.default_rng(SEED + 1)
cls_idx     = {c: np.where(val_labels == c)[0] for c in range(N_CLASSES)}

same_pairs = []
for c in range(N_CLASSES):
    idxs = cls_idx[c]
    for _ in range(N_PAIRS_EACH // N_CLASSES):
        i, j = rng_pairs.choice(idxs, size=2, replace=False)
        same_pairs.append((int(i), int(j)))

diff_pairs = []
for _ in range(N_PAIRS_EACH):
    c1, c2 = rng_pairs.choice(N_CLASSES, size=2, replace=False)
    i = int(rng_pairs.choice(cls_idx[c1]))
    j = int(rng_pairs.choice(cls_idx[c2]))
    diff_pairs.append((i, j))

all_pairs          = same_pairs + diff_pairs
pair_labels_binary = np.array([1]*len(same_pairs) + [0]*len(diff_pairs))
same_sl            = slice(0, len(same_pairs))
print(f'Pairs: {len(same_pairs)} same-class + {len(diff_pairs)} diff-class = {len(all_pairs)} total')

# ── Compute similarities ─────────────────────────────────────────────────────
print('Computing Phase 0 similarities (ground truth)...')
layer_sims_p0,  _,        h8_sims_p0  = compute_pair_sims(
    backbone_p0, None, val_ds_raw, all_pairs, DEVICE)
mean_traj_p0 = layer_sims_p0.mean(axis=1)   # primary ground truth

print('Computing Model A similarities...')
layer_sims_a, z_sims_a, h8_sims_a = compute_pair_sims(
    backbone_a, meta_enc_a, val_ds_raw, all_pairs, DEVICE)

print('Computing Model B similarities...')
layer_sims_b, z_sims_b, h8_sims_b = compute_pair_sims(
    backbone_b, meta_enc_b, val_ds_raw, all_pairs, DEVICE)

# ── Summary table ─────────────────────────────────────────────────────────────
print()
print(f'  {"Metric":<28} | {"Model A (Aug)":>14} | {"Model B (Hebb)":>14} | {"Phase 0":>10}')
print('  ' + '-'*72)
for name, za, zb, zp in [
    ('mean z-sim (same-class)',    z_sims_a[same_sl].mean(), z_sims_b[same_sl].mean(), None),
    ('mean z-sim (diff-class)',    z_sims_a[N_PAIRS_EACH:].mean(), z_sims_b[N_PAIRS_EACH:].mean(), None),
    ('mean h8-sim (same)',         h8_sims_a[same_sl].mean(), h8_sims_b[same_sl].mean(), h8_sims_p0[same_sl].mean()),
    ('mean h8-sim (diff)',         h8_sims_a[N_PAIRS_EACH:].mean(), h8_sims_b[N_PAIRS_EACH:].mean(), h8_sims_p0[N_PAIRS_EACH:].mean()),
    ('mean traj-sim (same)',       layer_sims_a[same_sl].mean(), layer_sims_b[same_sl].mean(), layer_sims_p0[same_sl].mean()),
    ('mean traj-sim (diff)',       layer_sims_a[N_PAIRS_EACH:].mean(), layer_sims_b[N_PAIRS_EACH:].mean(), layer_sims_p0[N_PAIRS_EACH:].mean()),
]:
    sp = f'{zp:>10.3f}' if zp is not None else f'{"--":>10}'
    print(f'  {name:<28} | {za:>14.3f} | {zb:>14.3f} | {sp}')

In [ ]:
# Section 6 — Accuracy
# Model A has no class signal; lower accuracy is expected and not a failure.
# Model B preserves classification ability (Phase 0 fc weights preserved).

print('Val accuracy (10-class CIFAR-10):')
print(f'  Phase 0 baseline  : {p0_acc_ref:.4f}  [cross-entropy baseline]')
print(f'  Model A (Aug)     : {acc_a:.4f}  [k-NN; trained without class labels]')
print(f'  Model B (Hebbian) : {acc_b:.4f}  [logit; Phase 0 fc preserved]')
if nb01_acc is not None:
    print(f'  nb01 reference    : {nb01_acc:.4f}  [class-label CTLS]')
else:
    print(f'  nb01 reference    : 0.7580  [hardcoded from prior run]')
print()
print('Note: accuracy is a secondary metric for this ablation.')
print('  Model A trained without class labels; k-NN reflects representation quality.')
print('  T2/T3/T4/T5 are the primary validation criteria.')

## Section 7 — Proxy ρ: z-sim vs. Phase 0 Trajectory Ground Truth

All Spearman ρ computed against `mean_traj_p0` (Phase 0 baseline trajectory similarities) — independent of both models.

**T2 signal**: does z add information beyond h₈? `ρ(z-sim, mean_traj_p0) - ρ(h₈-sim, mean_traj_p0) ≥ 0.05`

**T2-within** (most important): same signal restricted to same-class pairs — tests whether z distinguishes which same-class pairs share circuits.

In [ ]:
# 7a. Overall rho: z-sim and h8-sim vs. Phase 0 ground truth
rho_z_a,  _ = stats.spearmanr(z_sims_a,  mean_traj_p0)
rho_h8_a, _ = stats.spearmanr(h8_sims_a, mean_traj_p0)
rho_z_b,  _ = stats.spearmanr(z_sims_b,  mean_traj_p0)
rho_h8_b, _ = stats.spearmanr(h8_sims_b, mean_traj_p0)

t2_gap_a = rho_z_a - rho_h8_a
t2_gap_b = rho_z_b - rho_h8_b
t2_pass_a = t2_gap_a >= 0.05
t2_pass_b = t2_gap_b >= 0.05

print('7a. Overall rho vs. Phase 0 mean trajectory similarity [ground truth]')
print(f'  {"Metric":<38} | {"Model A":>9} | {"Model B":>9}')
print('  ' + '-'*60)
print(f'  {"rho(z-sim, mean_traj_p0)":<38} | {rho_z_a:>9.3f} | {rho_z_b:>9.3f}')
print(f'  {"rho(h8-sim, mean_traj_p0)":<38} | {rho_h8_a:>9.3f} | {rho_h8_b:>9.3f}')
print(f'  {"T2 gap = z - h8":<38} | {t2_gap_a:>+9.3f} | {t2_gap_b:>+9.3f}')
print(f'  {"T2 pass? (gap >= 0.05)":<38} | {str(t2_pass_a):>9} | {str(t2_pass_b):>9}')

In [ ]:
# 7b. T2-within: same-class pairs only
z_a_same   = z_sims_a[same_sl]
z_b_same   = z_sims_b[same_sl]
h8_a_same  = h8_sims_a[same_sl]
h8_b_same  = h8_sims_b[same_sl]
gt_same    = mean_traj_p0[same_sl]

rho_z_a_w,  _ = stats.spearmanr(z_a_same,  gt_same)
rho_h8_a_w, _ = stats.spearmanr(h8_a_same, gt_same)
rho_z_b_w,  _ = stats.spearmanr(z_b_same,  gt_same)
rho_h8_b_w, _ = stats.spearmanr(h8_b_same, gt_same)

t2w_gap_a = rho_z_a_w - rho_h8_a_w
t2w_gap_b = rho_z_b_w - rho_h8_b_w

mu_za_w,  lo_za_w,  hi_za_w  = bootstrap_spearmanr(z_a_same,  gt_same, seed=20)
mu_h8a_w, lo_h8a_w, hi_h8a_w = bootstrap_spearmanr(h8_a_same, gt_same, seed=21)
mu_zb_w,  lo_zb_w,  hi_zb_w  = bootstrap_spearmanr(z_b_same,  gt_same, seed=22)
mu_h8b_w, lo_h8b_w, hi_h8b_w = bootstrap_spearmanr(h8_b_same, gt_same, seed=23)

_, p_t2w_a, _ = permutation_test_delta_rho(h8_a_same, gt_same, z_a_same, seed=30)
_, p_t2w_b, _ = permutation_test_delta_rho(h8_b_same, gt_same, z_b_same, seed=31)

t2w_pass_a = (t2w_gap_a >= 0.03) and (p_t2w_a < 0.05)
t2w_pass_b = (t2w_gap_b >= 0.03) and (p_t2w_b < 0.05)

print('7b. T2-within: rho vs. Phase 0 traj-sim -- same-class pairs only (n=500)')
print(f'  {"Metric":<40} | {"Model A":>16} | {"Model B":>16}')
print('  ' + '-'*76)
print(f'  {"rho(z-sim, gt) [z-sim]":<40} | {mu_za_w:6.3f} [{lo_za_w:.3f},{hi_za_w:.3f}] '
      f'| {mu_zb_w:6.3f} [{lo_zb_w:.3f},{hi_zb_w:.3f}]')
print(f'  {"rho(h8-sim, gt) [baseline]":<40} | {mu_h8a_w:6.3f} [{lo_h8a_w:.3f},{hi_h8a_w:.3f}] '
      f'| {mu_h8b_w:6.3f} [{lo_h8b_w:.3f},{hi_h8b_w:.3f}]')
print(f'  {"T2-within gap (z - h8)":<40} | {t2w_gap_a:+16.3f} | {t2w_gap_b:+16.3f}')
print(f'  {"permutation p (one-sided)":<40} | {p_t2w_a:>16.4f} | {p_t2w_b:>16.4f}')
print(f'  {"T2-within PASS (gap>=0.03, p<0.05)?":<40} | {str(t2w_pass_a):>16} | {str(t2w_pass_b):>16}')

In [ ]:
# 7c. Per-layer gap: mean(same) - mean(diff) for each layer
def layer_gap(lsims, n_same):
    return lsims[:n_same].mean(0) - lsims[n_same:].mean(0)

gap_p0 = layer_gap(layer_sims_p0, len(same_pairs))
gap_a  = layer_gap(layer_sims_a,  len(same_pairs))
gap_b  = layer_gap(layer_sims_b,  len(same_pairs))

print('7c. Per-layer gap: mean(same-class) - mean(diff-class) per layer')
print('    Layers with gap > 0.05 that Phase 0 lacks = new circuit separation.')
print(f'  {"Layer":<6} | {"Phase 0":>9} | {"Model A":>9} | {"Model B":>9} | note')
print('  ' + '-'*58)
for l in range(L):
    note = ''
    if gap_a[l] > 0.05 and gap_p0[l] <= 0.05:
        note += ' A-gain'
    if gap_b[l] > 0.05 and gap_p0[l] <= 0.05:
        note += ' B-gain'
    print(f'  {l+1:<6} | {gap_p0[l]:>9.3f} | {gap_a[l]:>9.3f} | {gap_b[l]:>9.3f} |{note}')

n_early_a = sum(1 for l in range(4) if gap_a[l] > 0.05)
n_early_b = sum(1 for l in range(4) if gap_b[l] > 0.05)
n_early_p0 = sum(1 for l in range(4) if gap_p0[l] > 0.05)
print(f'Early layers (1-4) with gap > 0.05: Phase0={n_early_p0} ModelA={n_early_a} ModelB={n_early_b}')

In [ ]:
# 7d. Per-layer rho profile: rho(z-sim, layer_sims[:, l]) for each l
pl_rho_a = [stats.spearmanr(z_sims_a, layer_sims_a[:, l])[0] for l in range(L)]
pl_rho_b = [stats.spearmanr(z_sims_b, layer_sims_b[:, l])[0] for l in range(L)]

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(range(1, L+1), pl_rho_a, 'o-', label='Model A (Aug)', color='steelblue')
ax.plot(range(1, L+1), pl_rho_b, 's-', label='Model B (Hebbian)', color='darkorange')
ax.axhline(0.7, color='red', linestyle='--', linewidth=0.8, label='spike threshold')
ax.set_xlabel('Trajectory layer'); ax.set_ylabel('Spearman rho')
ax.set_title('7d. Per-layer rho profile: rho(z-sim, layer_sims[:, l])')
ax.legend(); ax.set_xticks(range(1, L+1))
plt.tight_layout(); plt.show()

print('Per-layer rho values:')
print(f'  {"Layer":<6} | {"Model A":>9} | {"Model B":>9}')
for l in range(L):
    print(f'  {l+1:<6} | {pl_rho_a[l]:>9.3f} | {pl_rho_b[l]:>9.3f}')

max_rho_a = max(pl_rho_a); max_rho_b = max(pl_rho_b)
others_a  = [r for r in pl_rho_a if r != max_rho_a]
others_b  = [r for r in pl_rho_b if r != max_rho_b]
t5_pass_a = not (max_rho_a > 0.7 and all(r < 0.4 for r in others_a))
t5_pass_b = not (max_rho_b > 0.7 and all(r < 0.4 for r in others_b))
print(f'T5: max_rho_a={max_rho_a:.3f} T5_pass_a={t5_pass_a} | max_rho_b={max_rho_b:.3f} T5_pass_b={t5_pass_b}')

In [ ]:
# 7e. Augmentation invariance: z-sim between two augmented views of same image
N_AUG_IMGS = 200
rng_aug    = np.random.default_rng(SEED + 5)
aug_img_idx = rng_aug.choice(len(val_ds_raw), N_AUG_IMGS, replace=False)
raw_val_ds  = CIFAR10(root=DATA_DIR, train=False, transform=None, download=False)

def aug_zsim(backbone, meta_enc, img_indices):
    backbone.eval(); meta_enc.eval()
    sims = []
    with torch.no_grad():
        for idx in img_indices:
            pil, _ = raw_val_ds[int(idx)]
            v1 = ssl_aug(pil).unsqueeze(0).to(DEVICE)
            v2 = ssl_aug(pil).unsqueeze(0).to(DEVICE)
            _, t1 = backbone(v1); _, t2 = backbone(v2)
            z1 = meta_enc(t1); z2 = meta_enc(t2)
            sims.append((z1 @ z2.T).item())
    return np.array(sims)

aug_sims_a = aug_zsim(backbone_a, meta_enc_a, aug_img_idx)
aug_sims_b = aug_zsim(backbone_b, meta_enc_b, aug_img_idx)

print('7e. Augmentation invariance: z-sim(view1, view2) for same image')
print(f'  Model A (trained for this): mean={aug_sims_a.mean():.3f} std={aug_sims_a.std():.3f}')
print(f'  Model B (not trained for this): mean={aug_sims_b.mean():.3f} std={aug_sims_b.std():.3f}')
print(f'  Model A same-class z-sim (reference): {z_sims_a[same_sl].mean():.3f}')
print()
print('  Model A augmentation sim >> same-class sim means views are tightly grouped.')

In [ ]:
# 7f. Cross-class z-neighbors (T4): fraction of top-20 neighbors crossing class boundaries
N_PROBE  = 1000
TOP_K_NN = 20
rng_probe = np.random.default_rng(SEED + 6)
probe_idx = rng_probe.choice(len(val_ds_raw), N_PROBE, replace=False)

def compute_knn_cross_class(backbone, meta_enc, val_ds, probe_idx, device, k=20, batch_size=256):
    backbone.eval(); meta_enc.eval()
    # Extract z for all val images
    dl = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=0)
    zs, lbls = [], []
    with torch.no_grad():
        for x, y in dl:
            _, traj = backbone(x.to(device))
            zs.append(meta_enc(traj).cpu())
            lbls.extend(y.tolist())
    zs   = torch.cat(zs)        # [N_val, D]
    lbls = np.array(lbls)
    probe_z   = zs[probe_idx]   # [N_probe, D]
    probe_lbl = lbls[probe_idx]
    sims = probe_z @ zs.T       # [N_probe, N_val]
    # Mask self
    for r, c in enumerate(probe_idx):
        sims[r, c] = -1.0
    topk_idx = sims.topk(k, dim=1).indices.numpy()  # [N_probe, k]
    topk_lbl = lbls[topk_idx]   # [N_probe, k]
    cross = (topk_lbl != probe_lbl[:, None]).mean()
    return float(cross), topk_idx, lbls

cross_a, topk_a, val_lbls_a = compute_knn_cross_class(
    backbone_a, meta_enc_a, val_ds_raw, probe_idx, DEVICE)
cross_b, topk_b, val_lbls_b = compute_knn_cross_class(
    backbone_b, meta_enc_b, val_ds_raw, probe_idx, DEVICE)

t4_pass_a = cross_a > 0.10
t4_pass_b = cross_b > 0.10

print('7f. Cross-class z-neighbors: fraction of top-20 neighbors crossing class boundaries')
print(f'  Model A: cross-class rate = {cross_a:.3f}  T4 pass (>0.10)? {t4_pass_a}')
print(f'  Model B: cross-class rate = {cross_b:.3f}  T4 pass (>0.10)? {t4_pass_b}')

In [ ]:
# Section 8 — Bootstrap CIs and Permutation Tests
print('Bootstrap 95% CIs (n_boot=2000):')
boot_rows = [
    ('rho(z_a, mean_traj_p0)',        z_sims_a,        mean_traj_p0, 40),
    ('rho(z_b, mean_traj_p0)',        z_sims_b,        mean_traj_p0, 41),
    ('rho(h8_a, mean_traj_p0)',       h8_sims_a,       mean_traj_p0, 42),
    ('rho(h8_b, mean_traj_p0)',       h8_sims_b,       mean_traj_p0, 43),
    ('rho(z_a, gt) same-class',       z_a_same,        gt_same,      44),
    ('rho(z_b, gt) same-class',       z_b_same,        gt_same,      45),
    ('rho(h8_a, gt) same-class',      h8_a_same,       gt_same,      46),
    ('rho(h8_b, gt) same-class',      h8_b_same,       gt_same,      47),
    ('rho(z_a, labels)',              z_sims_a,        pair_labels_binary.astype(float), 48),
    ('rho(z_b, labels)',              z_sims_b,        pair_labels_binary.astype(float), 49),
]
print(f'  {"Metric":<38} | rho    | 95% CI')
print('  ' + '-'*60)
for name, x, y, seed in boot_rows:
    mu, lo, hi = bootstrap_spearmanr(x, y, seed=seed)
    print(f'  {name:<38} | {mu:6.3f} | [{lo:.3f}, {hi:.3f}]')

print()
print('Permutation tests (n_perm=2000):')
perm_results = []
tests = [
    ('Test 1: z_a > h8_a vs mean_traj_p0',     h8_sims_a, mean_traj_p0, z_sims_a,  1),
    ('Test 2: z_b > h8_b vs mean_traj_p0',     h8_sims_b, mean_traj_p0, z_sims_b,  2),
    ('Test 3: z_b > z_a vs mean_traj_p0',      z_sims_a,  mean_traj_p0, z_sims_b,  3),
    ('Test 4: z_b > z_a vs mean_traj_p0 same', z_a_same,  gt_same,      z_b_same,  4),
]
nulls = []
for name, x1, y, x2, seed in tests:
    delta, p, null = permutation_test_delta_rho(x1, y, x2, seed=seed)
    perm_results.append((name, delta, p, null))
    nulls.append((null, delta, p, name))
    print(f'  {name}: delta={delta:+.3f}  p={p:.4f}')

fig, axes = plt.subplots(1, 4, figsize=(18, 3))
for ax, (null, delta, p, name) in zip(axes, nulls):
    ax.hist(null, bins=40, color='lightgray', edgecolor='gray')
    ax.axvline(delta, color='red', linewidth=2, label=f'delta={delta:+.3f}')
    ax.axvline(0, color='black', linewidth=0.8, linestyle=':')
    ax.set_title(name.split(':')[0] + f'\np={p:.4f}', fontsize=8)
    ax.set_xlabel('delta rho'); ax.legend(fontsize=7)
plt.suptitle('Permutation null distributions (Phase 0 GT)', fontsize=10)
plt.tight_layout(); plt.show()

In [ ]:
# Section 9a — UMAP
if not _UMAP_OK:
    print('Skipping UMAP (umap-learn not available).')
else:
    N_VIZ = 2000
    rng_viz = np.random.default_rng(SEED + 7)
    viz_idx = rng_viz.choice(len(val_ds_raw), N_VIZ, replace=False)
    viz_labels = val_labels[viz_idx]
    viz_sub = torch.utils.data.Subset(val_ds_raw, viz_idx.tolist())
    viz_dl  = DataLoader(viz_sub, batch_size=256, shuffle=False, num_workers=0)

    def extract_z_feats(backbone, meta_enc, dl, device):
        backbone.eval()
        if meta_enc is not None:
            meta_enc.eval()
        zs, h8s = [], []
        with torch.no_grad():
            for x, _ in dl:
                _, traj = backbone(x.to(device))
                h8s.append(F.normalize(traj[-1], dim=-1).cpu())
                if meta_enc is not None:
                    zs.append(meta_enc(traj).cpu())
        return (torch.cat(zs).numpy() if zs else None,
                torch.cat(h8s).numpy())

    z_a_viz,  h8_a_viz  = extract_z_feats(backbone_a,  meta_enc_a,  viz_dl, DEVICE)
    z_b_viz,  h8_b_viz  = extract_z_feats(backbone_b,  meta_enc_b,  viz_dl, DEVICE)
    _,        h8_p0_viz = extract_z_feats(backbone_p0, None,        viz_dl, DEVICE)

    def run_umap(feats, seed=SEED):
        from sklearn.decomposition import PCA
        feats = PCA(n_components=min(50, feats.shape[1])).fit_transform(feats)
        return umap.UMAP(n_components=2, random_state=seed, n_neighbors=15).fit_transform(feats)

    emb_p0 = run_umap(h8_p0_viz)
    emb_a  = run_umap(z_a_viz)
    emb_b  = run_umap(z_b_viz)

    # For Model B: also color by cluster assignment (if images in cluster_index)
    p0_img_set = set(image_indices_arr.tolist())

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    titles = ['Phase 0 (h8, PCA+UMAP)', 'Model A z (Aug)', 'Model B z (Hebbian)']
    for ax, emb, title in zip(axes, [emb_p0, emb_a, emb_b], titles):
        sc = ax.scatter(emb[:, 0], emb[:, 1], c=viz_labels, cmap='tab10',
                        s=4, alpha=0.6)
        ax.set_title(title, fontsize=10)
        ax.axis('off')
    plt.colorbar(sc, ax=axes[-1], label='CIFAR-10 class')
    plt.suptitle('UMAP: color by class (2000 val images)', fontsize=11)
    plt.tight_layout(); plt.show()
    print('9a: If Model B shows within-class sub-clusters, Hebbian clustering produced'
          ' finer-grained organization than class identity alone.')

In [ ]:
# Section 9b — z-neighborhood grid (8 anchors, top-6 neighbors)
import torchvision.transforms.functional as TF

N_ANCHORS  = 8
N_NEIGHBORS = 6
rng_anch   = np.random.default_rng(SEED + 8)
# Pick anchors: 1 per class for diversity
anchor_idx = []
for c in range(N_ANCHORS):
    anchor_idx.append(int(rng_anch.choice(cls_idx[c % N_CLASSES])))

raw_val_ds2 = CIFAR10(root=DATA_DIR, train=False, transform=None, download=False)

def get_z_all(backbone, meta_enc, val_ds, device, batch_size=256):
    backbone.eval(); meta_enc.eval()
    dl = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=0)
    zs, lbls = [], []
    with torch.no_grad():
        for x, y in dl:
            _, traj = backbone(x.to(device))
            zs.append(meta_enc(traj).cpu())
            lbls.extend(y.tolist())
    return torch.cat(zs), np.array(lbls)

z_all_a, lbls_all = get_z_all(backbone_a, meta_enc_a, val_ds_raw, DEVICE)
z_all_b, _        = get_z_all(backbone_b, meta_enc_b, val_ds_raw, DEVICE)

def show_grid(z_all, lbls, anchor_idx, ax_row, title_prefix):
    for col, anc in enumerate(anchor_idx):
        sims = (z_all[anc].unsqueeze(0) @ z_all.T).squeeze()
        sims[anc] = -1.0
        nn_idx = sims.topk(N_NEIGHBORS).indices.tolist()
        row_imgs = [anc] + nn_idx
        for row, img_i in enumerate(row_imgs):
            ax = ax_row[col][row]
            pil, lbl = raw_val_ds2[img_i]
            ax.imshow(np.array(pil))
            ax.axis('off')
            if row == 0:
                ax.set_title(f'anchor\ncls={lbl}', fontsize=6)
            else:
                same = lbl == lbls[anc]
                for sp in ax.spines.values():
                    sp.set_edgecolor('green' if same else 'blue')
                    sp.set_linewidth(2)
                ax.set_visible(True)

fig = plt.figure(figsize=(20, 6))
# Model A row
gs = gridspec.GridSpec(2, N_ANCHORS * (N_NEIGHBORS+1), hspace=0.3)
row_a = [[fig.add_subplot(gs[0, col*(N_NEIGHBORS+1)+r]) for r in range(N_NEIGHBORS+1)]
         for col in range(N_ANCHORS)]
row_b = [[fig.add_subplot(gs[1, col*(N_NEIGHBORS+1)+r]) for r in range(N_NEIGHBORS+1)]
         for col in range(N_ANCHORS)]
show_grid(z_all_a, lbls_all, anchor_idx, row_a, 'Model A')
show_grid(z_all_b, lbls_all, anchor_idx, row_b, 'Model B')
fig.text(0.01, 0.75, 'Model A', va='center', rotation='vertical', fontsize=10)
fig.text(0.01, 0.25, 'Model B', va='center', rotation='vertical', fontsize=10)
plt.suptitle('9b. z-neighborhood grid (anchor | top-6 neighbors; green=same class, blue=diff)', fontsize=10)
plt.tight_layout(); plt.show()

In [ ]:
# Section 9c — Within-class silhouette using k-means sub-clusters (k=4 per class)
# Higher sub-cluster silhouette for Model B than A = Hebbian organization finer-grained.

N_SUB = 4   # sub-clusters per class

def within_class_silhouette(z_all, lbls, n_sub=4, seed=SEED):
    z_np = z_all.numpy()
    all_labels = []
    all_feats  = []
    for c in range(N_CLASSES):
        mask = (lbls == c)
        feats_c = z_np[mask]
        km_c = KMeans(n_clusters=n_sub, n_init=5, random_state=seed).fit(feats_c)
        all_feats.append(feats_c)
        # Unique sub-cluster labels per class
        all_labels.append(km_c.labels_ + c * n_sub)
    all_feats  = np.concatenate(all_feats)
    all_labels = np.concatenate(all_labels)
    return silhouette_score(all_feats, all_labels)

sil_a = within_class_silhouette(z_all_a, lbls_all)
sil_b = within_class_silhouette(z_all_b, lbls_all)

print('9c. Within-class sub-cluster silhouette (k=4 per class)')
print(f'  Model A (Aug):     {sil_a:.4f}')
print(f'  Model B (Hebbian): {sil_b:.4f}')
print()
if sil_b > sil_a:
    print('  Model B > Model A: Hebbian clustering produced finer within-class organization.')
else:
    print('  Model A >= Model B: Augmentation did not produce less within-class structure.')

In [ ]:
# Section 10 — Non-Triviality Tests
# T1: Collapse broken  -- within-image z-sim > same-class z-sim
# T2: z adds over h8   -- rho(z, mean_traj_p0) - rho(h8, mean_traj_p0) >= 0.05
#                          OR T2-within gap >= 0.03 AND p < 0.05
# T3: Early-layer engagement (layers 1-4 show gap > 0.05 that Phase 0 lacks)
# T4: Cross-class z-neighbors > 10%
# T5: Per-layer rho profile not dominated by layer 8

# ── T1 ───────────────────────────────────────────────────────────────────────
def within_img_zsim(backbone, meta_enc, val_ds, n_imgs=200, device=DEVICE):
    rng = np.random.default_rng(SEED + 9)
    idxs = rng.choice(len(val_ds), n_imgs, replace=False)
    raw  = CIFAR10(root=DATA_DIR, train=False, transform=None, download=False)
    sims = []
    backbone.eval(); meta_enc.eval()
    with torch.no_grad():
        for idx in idxs:
            pil, _ = raw[int(idx)]
            v1 = val_tf(pil).unsqueeze(0).to(device)
            v2 = val_tf(pil).unsqueeze(0).to(device)
            _, t1 = backbone(v1); _, t2 = backbone(v2)
            z1 = meta_enc(t1); z2 = meta_enc(t2)
            sims.append((z1 @ z2.T).item())
    return np.array(sims)

wi_a = within_img_zsim(backbone_a, meta_enc_a, val_ds_raw)
wi_b = within_img_zsim(backbone_b, meta_enc_b, val_ds_raw)
sa_a = z_sims_a[same_sl].mean()
sa_b = z_sims_b[same_sl].mean()
t1_gap_a = wi_a.mean() - sa_a
t1_gap_b = wi_b.mean() - sa_b
t1_pass_a = t1_gap_a > 0
t1_pass_b = t1_gap_b > 0

# ── T2 (combined) ────────────────────────────────────────────────────────────
t2_combined_a = t2_pass_a or t2w_pass_a
t2_combined_b = t2_pass_b or t2w_pass_b

# ── T3 ───────────────────────────────────────────────────────────────────────
improved_early_a = [l+1 for l in range(4) if gap_a[l] > 0.05 and gap_p0[l] <= 0.05]
improved_early_b = [l+1 for l in range(4) if gap_b[l] > 0.05 and gap_p0[l] <= 0.05]
t3_pass_a = (n_early_a > n_early_p0) or (len(improved_early_a) >= 1)
t3_pass_b = (n_early_b > n_early_p0) or (len(improved_early_b) >= 1)

# ── Summary ──────────────────────────────────────────────────────────────────
PASS = 'PASS'
FAIL = 'FAIL'
tests_10 = [
    ('T1', 'Collapse broken',
     'within > same-class (gap > 0)',
     t1_pass_a, t1_pass_b,
     f'A: within={wi_a.mean():.3f} same={sa_a:.3f} gap={t1_gap_a:+.3f}',
     f'B: within={wi_b.mean():.3f} same={sa_b:.3f} gap={t1_gap_b:+.3f}'),
    ('T2', 'z adds over h8',
     'rho gap>=0.05 overall OR gap>=0.03 same-class (p<0.05)',
     t2_combined_a, t2_combined_b,
     f'A: overall={t2_gap_a:+.3f} within={t2w_gap_a:+.3f} p={p_t2w_a:.3f}',
     f'B: overall={t2_gap_b:+.3f} within={t2w_gap_b:+.3f} p={p_t2w_b:.3f}'),
    ('T3', 'Early-layer engagement',
     '>= 1 early layer (1-4) with gap > 0.05 not in Phase 0',
     t3_pass_a, t3_pass_b,
     f'A: new gains at layers {improved_early_a}',
     f'B: new gains at layers {improved_early_b}'),
    ('T4', 'Cross-class z-neighbors',
     '> 10% top-20 neighbors cross class boundaries',
     t4_pass_a, t4_pass_b,
     f'A: {cross_a:.3f}',
     f'B: {cross_b:.3f}'),
    ('T5', 'Layer profile not dominated',
     'No single layer rho > 0.7 with all others < 0.4',
     t5_pass_a, t5_pass_b,
     f'A: max={max_rho_a:.3f}',
     f'B: max={max_rho_b:.3f}'),
]

print('=' * 76)
print('SECTION 10: NON-TRIVIALITY TEST SUMMARY')
print('=' * 76)
print(f'  {"Test":<5} {"Name":<26} {"Model A":>8} {"Model B":>8}  Criterion')
print('  ' + '-'*76)
n_pass_a = n_pass_b = 0
for tid, name, crit, pa, pb, ev_a, ev_b in tests_10:
    n_pass_a += int(pa); n_pass_b += int(pb)
    print(f'  {tid:<5} {name:<26} {PASS if pa else FAIL:>8} {PASS if pb else FAIL:>8}  {crit}')
    print(f'        Evidence: {ev_a}')
    print(f'                  {ev_b}')
print(f'  Total passed: Model A = {n_pass_a}/5,  Model B = {n_pass_b}/5')
print()
for n_pass, label in [(n_pass_a, 'Model A'), (n_pass_b, 'Model B')]:
    if n_pass >= 4:
        print(f'  {label}: STRONG evidence of multi-layer circuit organization.')
    elif n_pass >= 2:
        print(f'  {label}: PARTIAL success. Review failed tests and cluster diagnostics.')
    else:
        print(f'  {label}: WEAK results. Check cluster purity (Sec 3b) for Model B;'
              f' check augmentation for Model A.')

In [ ]:
# Section 11 — Summary Results Table
# nb01 reference: hardcoded from prior run (class-label CTLS)
NB01_RHO_LABELS = 0.836
NB01_RHO_TRAJ   = 0.797
NB01_SIL        = 0.819

rho_z_p0,  _ = stats.spearmanr(
    layer_sims_p0.mean(1),  # Phase 0 has no z; use h8 as proxy
    mean_traj_p0)
rho_h8_p0, _ = stats.spearmanr(h8_sims_p0, mean_traj_p0)

mu_za,  lo_za,  hi_za  = bootstrap_spearmanr(z_sims_a,  mean_traj_p0, seed=60)
mu_zb,  lo_zb,  hi_zb  = bootstrap_spearmanr(z_sims_b,  mean_traj_p0, seed=61)

nb01_rho_traj_live = None
if backbone_nb01 is not None:
    _, z_nb01_s, h8_nb01_s = compute_pair_sims(
        backbone_nb01, meta_enc_nb01, val_ds_raw, all_pairs, DEVICE)
    nb01_rho_traj_live, _ = stats.spearmanr(z_nb01_s, mean_traj_p0)

def fmt(v, fallback='--'):
    return f'{v:.3f}' if v is not None else fallback

nb01_rho_col = fmt(nb01_rho_traj_live) if nb01_rho_traj_live is not None else f'{NB01_RHO_TRAJ:.3f}*'

rows = [
    ('Val accuracy',             fmt(p0_acc_ref), fmt(acc_a)+'(knn)', fmt(acc_b), fmt(nb01_acc) if nb01_acc else '~0.758*'),
    ('rho(z, mean_traj_p0)',     fmt(rho_h8_p0)+' (h8)', f'{mu_za:.3f} [{lo_za:.3f},{hi_za:.3f}]', f'{mu_zb:.3f} [{lo_zb:.3f},{hi_zb:.3f}]', nb01_rho_col),
    ('rho(h8, mean_traj_p0)',    fmt(rho_h8_p0), fmt(rho_h8_a), fmt(rho_h8_b), '--'),
    ('T2 gap (z - h8) overall',  '--',            fmt(t2_gap_a),  fmt(t2_gap_b),  '--'),
    ('T2-within gap (same-cls)', '--',            fmt(t2w_gap_a), fmt(t2w_gap_b), '--'),
    ('Early layers gap>0.05/4',  str(n_early_p0), str(n_early_a), str(n_early_b), '--'),
    ('Within-image z-sim',       '--',            fmt(wi_a.mean()), fmt(wi_b.mean()), '--'),
    ('Same-class z-sim',         '--',            fmt(sa_a),       fmt(sa_b),       '--'),
    ('T1 gap (within - same)',   '--',            fmt(t1_gap_a),   fmt(t1_gap_b),   '--'),
    ('T4 cross-class rate',      '--',            fmt(cross_a),    fmt(cross_b),    '--'),
    ('Max per-layer rho (T5)',    '--',            fmt(max_rho_a),  fmt(max_rho_b),  '--'),
    ('Tests passed (X/5)',        '--',           f'{n_pass_a}/5',  f'{n_pass_b}/5', '--'),
]

W = [28, 20, 20, 20, 16]
header = ['Metric', 'Phase 0 baseline', 'Model A (Aug)', 'Model B (Hebbian)', 'nb01 ref']
sep = '  +' + '+'.join('-'*(w+2) for w in W) + '+'
hdr = '  |' + '|'.join(f' {h:<{w}} ' for h, w in zip(header, W)) + '|'
print('\n' + sep)
print(hdr)
print(sep)
for row in rows:
    print('  |' + '|'.join(f' {str(v):<{w}} ' for v, w in zip(row, W)) + '|')
print(sep)
print('  * = hardcoded from prior run')

## Appendix — Interpretation Guide

**T2-within passing** means z distinguishes which same-class pairs share circuits vs. which don't. This is the core CTLS claim: z captures circuit structure, not just class membership. T2-overall can pass trivially if z becomes a strong class classifier; T2-within cannot.

**Cluster purity > 0.85 for Model B** means Phase 0 trajectory clusters rediscovered class structure. Phase 1 will then degrade to a variant of nb01 (class-label contrastive learning), losing any advantage over Model A. If purity > 0.85, consider: (1) increasing k (200+ clusters), (2) using only intra-class trajectory distance for clustering, or (3) patching-derived pairs (same image, different augmentation weight, different circuit activation pattern).

**Why Model B uses lr=5e-4 in Phase 1** (vs 1e-3 for Model A): Phase 0 weights are already converged at a good classification optimum. A high learning rate would destroy the learned feature representations before the InfoNCE loss has time to form circuit clusters. Lower lr allows gentle perturbation of the feature space.

**What to try if both models fail T2**:
1. Longer Phase 0 training (>100 epochs) — richer trajectory space
2. More clusters (k=200+) — finer-grained Hebbian pairs
3. Patching-derived positive pairs — same image processed with different input    corruptions targeting specific layers
4. Explicit early-layer auxiliary loss — add a small cross-entropy head on layer 2    activations to ensure early layers carry discriminative signal

**What T4 cross-class rate measures**: if z ≈ h₈ (collapse), then z-space neighbors are almost entirely same-class (h₈ is class-discriminative). Cross-class rate > 10% means z has organized space around some criterion *other than* class membership — a necessary (but not sufficient) condition for circuit-level organization.

**nb01 reference numbers** in the summary table are hardcoded from a prior run and represent the class-label CTLS baseline (Option A). They are included to show the regime where class-label training collapses z ≈ h₈ (nb01 T2 gap is near zero or negative despite high ρ(z, labels)).